# A15 · Multi-touch attribution and randomized incrementality

Prerequisites: foundations 31–34; A14.

Evidence: **explicitly constructed mechanism experiments, not real ad logs**. This chapter does not claim industrial-system reproduction or online gains.

Learning order: predict → change one variable → run → explain.

> **English companion.** All instructional Markdown is translated. Experiment code, Chinese language examples, execution counts, and saved outputs remain as in the source edition; historical output text may therefore be Chinese. The initial setup cell anchors the working directory and Python import path for this companion's deeper folder; it does not change the experiment. Saved outputs are historical evidence, not a new run. Run from the top after preparing the required data and models.

In [ ]:
# English companions live one directory deeper; keep the original experiment code unchanged.
from pathlib import Path as _TranslationPath
import os as _translation_os
import sys as _translation_sys
_translation_root = next((p for p in [_TranslationPath.cwd(), *_TranslationPath.cwd().parents]
                          if (p / "reclab").is_dir()), None)
if _translation_root is None:
    raise RuntimeError("Open this notebook from inside the Search Recommendation Ads Lab checkout.")
_translation_os.chdir(_translation_root)
if str(_translation_root) not in _translation_sys.path:
    _translation_sys.path.insert(0, str(_translation_root))


## Problem and input/output contract

Last-click credits all revenue to the final touch; linear attribution splits it equally across touches. These accounting rules conserve totals but cannot answer what would happen without ads. In a separate constructed randomized experiment, assignment probability is known at .5 and paired potential outcomes use the same random number. Treatment changes conversion probability from .10 to .13. Compare observed mean difference with the known finite-population causal effect. There are no real propensities or real IPS/OPE results here.

Before running, predict which result the changed variable will affect and which invariants must hold.

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## Run the one-variable comparison

Inputs are small constructed examples visible in source. Output units and constraints are defined above. Run assertions to verify mathematics and accounting.

In [2]:
a=ac.attribution();print(a)
assert np.isclose(sum(a['last_click'].values()),250)
assert np.isclose(sum(a['linear'].values()),250)
assert a['last_click']!=a['linear']
print('随机估计与潜在结果差距',a['randomized_difference']-a['finite_population_effect'])

{'last_click': {'display': 250.0}, 'linear': {'search': 110.0, 'display': 140.0}, 'randomized_difference': 0.03510874558569296, 'finite_population_effect': 0.0315, 'n': 10000, 'assignment_probability': 0.5}
随机估计与潜在结果差距 0.003608745585692963


## Interpretation and limits

Attribution allocations may change sharply while total revenue stays fixed; randomized incrementality answers a different question. A finite-sample randomized estimate differs from known potential effects. Real experiments must prespecify randomization unit, confidence intervals, cross-group interference, and observation windows.

## Inspect source, then try an exercise

Switch to first-click while preserving total 250. Construct nonrandom delivery favoring high-intent users and explain why a simple mean difference no longer identifies incrementality.

In [3]:
import inspect
print(inspect.getsource(ac.attribution))

def attribution():
    paths=[(['search','display'],100.),(['display'],60.),(['search','search','display'],90.)]
    last={};linear={}
    for path,revenue in paths:
        last[path[-1]]=last.get(path[-1],0)+revenue
        for channel in path: linear[channel]=linear.get(channel,0)+revenue/len(path)
    assert abs(sum(last.values())-sum(linear.values()))<1e-9
    rng=np.random.default_rng(42); n=10000
    assignment=rng.integers(0,2,n); latent=rng.random(n)
    y0=latent<.10; y1=latent<.13; observed=np.where(assignment,y1,y0)
    return dict(last_click=last,linear=linear,randomized_difference=float(observed[assignment==1].mean()-observed[assignment==0].mean()),
                finite_population_effect=float((y1.astype(int)-y0).mean()),n=n,assignment_probability=.5)



## Sources and next step

A-RTB pp. 94–116; the known-randomization incrementality example is Lab-authored. This is an original Lab exercise, not a copy of the complete lecture text.

[Chinese GitHub chapter](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [Authors’ RTB slides](https://wnzhang.net/slides/cikm-16-rtb-full.pdf). The Chinese deck is not a complete modern advertising-engineering course. Historical descriptions of platforms are not presented as current market facts.

After the exercise, return to the ads learning path and continue to the next semantic unit.